In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("SalesbyCustomer.csv")

In [ ]:
df.head()

In [ ]:
import pandas as pd

# 1. 確保Date欄位轉換為 datetime 格式
df["Date"] = pd.to_datetime(df["Date"])

# 2. 計算整個資料集的最大交易日（基準日）
max_date = df["Date"].max()

# 3. 依據CustomerKey進行聚合計算
RFM_Data = (
    df.groupby("CustomerKey")
    .agg(
        # R: 最大交易日 與 該客戶最後交易日 的天數差
        R=("Date", lambda x: (max_date - x.max()).days),
        # F: 交易次數 (計算不重複發票張數，若同一張發票有多列明細也不會重複計算)
        F=("InvoNo", "nunique"),
        # M: 交易總Amount
        M=("Amount", "sum"),
    )
    .reset_index()
)

# 4. 將「CustomerKey」欄位名稱更改為「客戶」
RFM_Data = RFM_Data.rename(columns={"CustomerKey": "客戶"})

# 調整欄位順序（客戶, R, F, M）
RFM_Data = RFM_Data[["客戶", "R", "F", "M"]]

# 檢視成果
print(RFM_Data.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

# 複製原始資料
rfmn = RFM_Data.copy()

# 使用 StandardScaler 進行正規化
scaler = StandardScaler()
rfmn[['R', 'F', 'M']] = scaler.fit_transform(rfmn[['R', 'F', 'M']])

# 檢視正規化後的資料
print(rfmn.head())

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 提取 RFM 特徵欄位進行 K-means 計算
X = rfmn[['R', 'F', 'M']]

wcss = []
k_range = range(1, 11)

# 計算不同 k 值下的群內誤差平方和 (Inertia)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

# 繪製 Elbow Chart
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='--', color='b')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans

# 提取 RFM 特徵
X = rfmn[['R', 'F', 'M']]

# 執行 k-means 分群 (k = 4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfmn['Cluster'] = kmeans.fit_predict(X)

# 顯示各分群的數量分布
print("各分群客戶數量：")
print(rfmn['Cluster'].value_counts())

# 檢視部分結果
print("\n前 5 筆結果：")
print(rfmn.head())

In [ ]:
rfmn

In [ ]:
# 載入 Silhouette 套件，為驗證分組成效，需要執行 Silhouette 分析
from sklearn.metrics import silhouette_samples
silhouette=silhouette_samples(rfmn[["R","F","M"]], rfmn.Cluster)

In [ ]:
# 01 設定畫布尺寸
# 02 初始類別軸起始點
# 03 繪製 Silhouette 圖

k=4 #承繼前段 k-means 分組數
plt.figure(figsize=[12,8]) #01
x0=0 #02

#03
for i in range(k):
   
    group_slh=silhouette[rfmn.Cluster==i] # 取出 i 組 silhouette 分數表
    x=range(x0,x0+len(group_slh)) # 以 i 組序列範圍為類別軸 (x)

    y=-np.sort(-group_slh) # 將 i 組 silhouette 分數表依遞減排序，為數值軸 (y)
    yavg=np.average(group_slh) # 計算 i 組 silhouette 分數平均值，為數值軸參考線

    plt.fill_between(x,y,alpha=0.2,label=i) # 繪圖區填滿
 
    plt.plot([x0,x0+len(group_slh)],[yavg,yavg],'--') # 標註平均線
   
    plt.legend() # 繪製圖例
       
    x0=x0+len(group_slh) # 設定次組類別軸起始點

plt.title('Silhouette Score of optimal K',fontsize=20)
plt.xlabel('Item No',fontsize=15)

In [ ]:
# 將 rfmn 中的 Cluster 標籤複製回 RFM_Data
RFM_Data['Cluster'] = rfmn['Cluster']

# 檢視合併後的原始 RFM 資料
print(RFM_Data.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 使用 Seaborn 繪製成對散布圖，並依 Cluster 欄位進行著色分群
sns.pairplot(RFM_Data, vars=['R', 'F', 'M'], hue='Cluster', palette='tab10', diag_kind='kde')
plt.suptitle('RFM Pairwise Scatter Plot by Cluster', y=1.02)
plt.show()

In [ ]:
# 定義 Cluster 對應 Group 的對照表
group_mapping = {
    0: 'M',
    1: 'L',
    2: 'H',
    3: 'Missing'
}

# 新增 Group 欄位
RFM_Data['Group'] = RFM_Data['Cluster'].map(group_mapping)

# 檢視更新後的資料與各組別數量
print(RFM_Data['Group'].value_counts())
print("\n資料預覽：")
print(RFM_Data.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 使用 Seaborn 繪製成對散布圖，並依 Group 欄位進行著色分群
g = sns.pairplot(RFM_Data, vars=['R', 'F', 'M'], hue='Group', palette='tab10', diag_kind='kde')
plt.suptitle('RFM Pairwise Scatter Plot by Group', y=1.02)

# 匯出為透明背景的 PNG 格式
g.savefig('RFM_Pair.png', transparent=True, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 將 RFM_Data 匯出成 CSV 檔案，不保留 index
RFM_Data.to_csv('RFM_Res.csv', index=False)
print("已成功匯出至 RFM_Res.csv")